# Results overview

Простой вариант: читаем run-папки, собираем одну таблицу, сравниваем dtype и рисуем графики.

In [ ]:
from pathlib import Path
import json
import re
import ast
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

root = Path.cwd().resolve()
for p in [root] + list(root.parents):
    if (p / "pinn_model.py").exists():
        root = p
        break

out_dir = root / "report_results"
table_dir = out_dir / "tables"
fig_dir = out_dir / "figures"
log_dir = out_dir / "logs"
for p in [table_dir, fig_dir, log_dir]:
    p.mkdir(parents=True, exist_ok=True)

print("project root found")

## Функции

In [ ]:
def read_json(p):
    try:
        return json.loads(Path(p).read_text())
    except Exception:
        return {}


def num(x):
    if x is None:
        return np.nan
    if isinstance(x, str) and x.strip().lower() in ["", "nan", "none", "null"]:
        return np.nan
    try:
        return float(x)
    except Exception:
        return np.nan


def dtype_name(x):
    x = str(x).lower()
    if x in ["float32", "torch.float32"]:
        return "fp32"
    if x in ["float64", "torch.float64"]:
        return "fp64"
    if x in ["float16", "torch.float16"]:
        return "fp16"
    return x


def alpha_parts(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str) and x.strip().startswith("{"):
        try:
            return ast.literal_eval(x)
        except Exception:
            return {}
    return {}


def task_name_ru(task):
    d = {
        "helmholtz1d": "Гельмгольц",
        "burgers1d": "Бюргерс",
        "convection1d": "Конвекция",
        "heat1d": "Теплопроводность",
    }
    return d.get(str(task), str(task))


def fmt(x):
    x = num(x)
    if np.isnan(x):
        return "?"
    if abs(x - round(x)) < 1e-12:
        return str(int(round(x)))
    return f"{x:g}"


def param_from_summary(s):
    task = s.get("task_name", "")
    a = alpha_parts(s.get("alpha"))
    beta = s.get("beta", a.get("beta", np.nan))
    nu = s.get("nu", a.get("nu", np.nan))
    m = s.get("m", a.get("m", np.nan))
    alpha = s.get("alpha", a.get("alpha", np.nan))
    if isinstance(alpha, dict):
        alpha = alpha.get("alpha", np.nan)
    if task == "helmholtz1d":
        return "m", num(m)
    if task == "burgers1d":
        return "nu", num(nu)
    if task == "convection1d":
        return "beta", num(beta)
    if task == "heat1d":
        return "alpha", num(alpha)
    return "task", 0.0


def short_case(task, par, val):
    if par == "task":
        return task_name_ru(task)
    return f"{task_name_ru(task)}, {par}={fmt(val)}"


def variant_from_folder(name, task):
    x = str(name)
    x = re.sub(r"^run\d+_", "", x)
    x = re.sub(r"^exp\d+_r\d+_", "", x)
    x = re.sub(r"^overnight_r\d+_", "", x)
    x = re.sub(r"_(fp16|fp32|fp64|float16|float32|float64)_s?\d+$", "", x)
    x = re.sub(r"_(fp16|fp32|fp64|float16|float32|float64)_\d+$", "", x)
    x = re.sub(r"^(m\d+|nu\d+p?\d*|beta\d+p?\d*|alpha\d+p?\d*)_", "", x)
    if task and x.startswith(str(task) + "_"):
        x = x[len(str(task)) + 1:]
    if x.strip() == "":
        return "base"
    return x


def metrics_info(p):
    res = {"best_l2": np.nan, "final_l2": np.nan, "best_step": np.nan, "final_step": np.nan}
    if not p.exists():
        return res
    try:
        h = pd.read_csv(p)
    except Exception:
        return res
    if len(h) == 0 or "l2_error" not in h.columns:
        return res
    l2 = pd.to_numeric(h["l2_error"], errors="coerce")
    if l2.notna().any():
        i = l2.idxmin()
        res["best_l2"] = float(l2.loc[i])
        res["final_l2"] = float(l2.iloc[-1])
        if "step" in h.columns:
            step = pd.to_numeric(h["step"], errors="coerce")
            res["best_step"] = float(step.loc[i])
            res["final_step"] = float(step.iloc[-1])
    return res

## Загрузка run-папок

In [ ]:
rows = []
for p in root.rglob("summary.json"):
    parts = set(p.parts)
    if ".git" in parts or "report_results" in parts or "__pycache__" in parts:
        continue

    s = read_json(p)
    mpath = p.parent / "metrics.csv"
    mi = metrics_info(mpath)
    task = s.get("task_name", "")
    par, val = param_from_summary(s)
    dtype = dtype_name(s.get("dtype", ""))
    seed = num(s.get("seed"))
    best = num(s.get("best_l2_error", s.get("best_l2", np.nan)))
    final = num(s.get("final_l2_error", s.get("final_l2", np.nan)))
    if np.isnan(best):
        best = mi["best_l2"]
    if np.isnan(final):
        final = mi["final_l2"]
    err = s.get("error", s.get("exception", ""))
    variant = s.get("variant", variant_from_folder(p.parent.name, task))

    rows.append({
        "case": short_case(task, par, val),
        "task_name": task,
        "param": par,
        "param_value": val,
        "variant": variant,
        "dtype": dtype,
        "seed": seed,
        "best_l2_error": best,
        "final_l2_error": final,
        "best_step": num(s.get("best_step", mi["best_step"])),
        "final_step": num(s.get("final_step", mi["final_step"])),
        "elapsed_time": num(s.get("elapsed_time", s.get("time_sec", np.nan))),
        "hid_size": num(s.get("hid_size", s.get("hidden_dim", np.nan))),
        "num_layers": num(s.get("num_layers", np.nan)),
        "n_collocation": num(s.get("n_collocation", np.nan)),
        "n_ic": num(s.get("n_ic", np.nan)),
        "n_bc": num(s.get("n_bc", np.nan)),
        "adam_steps": num(s.get("adam_steps", np.nan)),
        "lr_adam": num(s.get("lr_adam", np.nan)),
        "lbfgs_steps": num(s.get("lbfgs_steps", np.nan)),
        "lbfgs_lr": num(s.get("lbfgs_lr", np.nan)),
        "lbfgs_max_iter": num(s.get("lbfgs_max_iter", np.nan)),
        "resample_every": num(s.get("resample_every", np.nan)),
        "error": err,
        "run_dir": str(p.parent.relative_to(root)),
        "metrics_path": str(mpath.relative_to(root)) if mpath.exists() else "",
    })

df = pd.DataFrame(rows)
for col in ["best_l2_error", "final_l2_error"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid_best = np.isfinite(df["best_l2_error"])
valid_final = np.isfinite(df["final_l2_error"]) | df["final_l2_error"].isna()
no_error = df["error"].fillna("").astype(str).str.len().eq(0)
df["is_valid"] = valid_best & valid_final & no_error
df["is_bad"] = (~df["is_valid"]) | (df["best_l2_error"] > 0.2) | (df["final_l2_error"] > 0.5)

def quality(row):
    if row["is_bad"]:
        return "bad"
    if row["best_l2_error"] <= 0.02 and row["final_l2_error"] <= 0.05:
        return "good"
    if row["best_l2_error"] <= 0.1 and row["final_l2_error"] <= 0.2:
        return "acceptable"
    return "unstable"

df["quality"] = df.apply(quality, axis=1)
df = df.sort_values(["task_name", "param_value", "variant", "dtype", "seed"])
df.to_csv(table_dir / "simple_all_runs.csv", index=False)

print("runs:", len(df))
print("valid:", int(df["is_valid"].sum()))
print("bad:", int(df["is_bad"].sum()))
display(df.head(20))

## Группировка

In [ ]:
keys = ["case", "task_name", "param", "param_value", "variant", "dtype"]
a = df.groupby(keys, dropna=False).agg(
    n_total=("run_dir", "count"),
    n_bad=("is_bad", "sum"),
).reset_index()

b = df[df["is_valid"]].groupby(keys, dropna=False).agg(
    n_valid=("run_dir", "count"),
    median_best_l2=("best_l2_error", "median"),
    mean_best_l2=("best_l2_error", "mean"),
    min_best_l2=("best_l2_error", "min"),
    max_best_l2=("best_l2_error", "max"),
    median_final_l2=("final_l2_error", "median"),
    seeds=("seed", lambda x: ", ".join(str(int(v)) for v in sorted(x.dropna().unique()))),
).reset_index()

g = a.merge(b, on=keys, how="left")
g["n_valid"] = g["n_valid"].fillna(0).astype(int)
g["bad_rate"] = g["n_bad"] / g["n_total"]
g.to_csv(table_dir / "simple_grouped.csv", index=False)

display(g.sort_values(["case", "variant", "dtype"]).head(30))

## FP32 / FP64

In [ ]:
pairs = []
for key, tmp in g[g["dtype"].isin(["fp32", "fp64"])].groupby(["case", "task_name", "param", "param_value", "variant"], dropna=False):
    d = {"case": key[0], "task_name": key[1], "param": key[2], "param_value": key[3], "variant": key[4]}
    for dt in ["fp32", "fp64"]:
        r = tmp[tmp["dtype"] == dt]
        if len(r):
            r = r.iloc[0]
            d[dt + "_n"] = int(r["n_valid"])
            d[dt + "_bad_rate"] = float(r["bad_rate"])
            d[dt + "_median"] = float(r["median_best_l2"]) if pd.notna(r["median_best_l2"]) else np.nan
            d[dt + "_min"] = float(r["min_best_l2"]) if pd.notna(r["min_best_l2"]) else np.nan
            d[dt + "_max"] = float(r["max_best_l2"]) if pd.notna(r["max_best_l2"]) else np.nan
        else:
            d[dt + "_n"] = 0
            d[dt + "_bad_rate"] = np.nan
            d[dt + "_median"] = np.nan
            d[dt + "_min"] = np.nan
            d[dt + "_max"] = np.nan
    if d["fp32_n"] == 0 or d["fp64_n"] == 0 or not np.isfinite(d["fp32_median"]) or not np.isfinite(d["fp64_median"]):
        d["ratio"] = np.nan
        d["result"] = "нет пары"
    else:
        d["ratio"] = d["fp64_median"] / d["fp32_median"]
        spread = False
        for dt in ["fp32", "fp64"]:
            mn = d[dt + "_min"]
            mx = d[dt + "_max"]
            if np.isfinite(mn) and mn > 0 and np.isfinite(mx) and mx / mn > 20:
                spread = True
        if spread or d["fp32_bad_rate"] >= 0.5 or d["fp64_bad_rate"] >= 0.5:
            d["result"] = "нестабильно"
        elif d["ratio"] <= 0.5:
            d["result"] = "fp64 лучше"
        elif d["ratio"] <= 0.8:
            d["result"] = "fp64 немного лучше"
        elif d["ratio"] < 1.25:
            d["result"] = "похоже"
        else:
            d["result"] = "fp32 лучше"
    pairs.append(d)

cmp_df = pd.DataFrame(pairs).sort_values(["result", "ratio"], na_position="last")
cmp_df.to_csv(table_dir / "simple_fp32_fp64_compare.csv", index=False)
display(cmp_df.head(40))

## Кейсы для графиков и конфиги

In [ ]:
selected = []
seen = set()

def add_rows(part, limit):
    for _, r in part.iterrows():
        k = str(r["case"])
        if k in seen:
            continue
        selected.append(r)
        seen.add(k)
        if len(selected) >= limit:
            break

pos = cmp_df[(cmp_df["result"].isin(["fp64 лучше", "fp64 немного лучше"])) & (cmp_df["fp32_n"] >= 1) & (cmp_df["fp64_n"] >= 1)]
pos = pos.sort_values("ratio")
add_rows(pos, 4)

sim = cmp_df[cmp_df["result"] == "похоже"].copy()
if len(sim):
    sim["dist"] = (sim["ratio"] - 1).abs()
    add_rows(sim.sort_values("dist"), 5)

neg = cmp_df[cmp_df["result"] == "fp32 лучше"].sort_values("ratio")
add_rows(neg, 6)

bad_fp32 = cmp_df[(cmp_df["fp32_bad_rate"] >= 0.5) & (cmp_df["fp64_bad_rate"] < 0.5)].sort_values("fp64_median")
add_rows(bad_fp32, 7)

selected = pd.DataFrame(selected)
selected.to_csv(table_dir / "simple_selected_cases.csv", index=False)

display(Markdown("### Кейсы для графиков"))
display(selected[["case", "variant", "fp32_n", "fp64_n", "fp32_median", "fp64_median", "ratio", "result"]])

print("Конфиги выбранных запусков")
for _, r in selected.iterrows():
    print("\n" + r["case"])
    print("variant:", r["variant"])
    tmp = df[(df["case"] == r["case"]) & (df["variant"] == r["variant"]) & (df["dtype"].isin(["fp32", "fp64", "fp16"]))]
    cols = ["dtype", "seed", "hid_size", "num_layers", "n_collocation", "n_ic", "n_bc", "adam_steps", "lr_adam", "lbfgs_steps", "lbfgs_lr", "lbfgs_max_iter", "resample_every", "best_l2_error", "final_l2_error"]
    print(tmp.sort_values(["dtype", "seed"])[cols].to_string(index=False))

## Основные графики

In [ ]:
if len(selected):
    labels = selected["case"].tolist()
    x = np.arange(len(selected))

    p = fig_dir / "simple_median_best_l2.png"
    fig, ax = plt.subplots(figsize=(max(8, len(selected) * 1.3), 4.5))
    ax.bar(x - 0.18, selected["fp32_median"], width=0.36, label="FP32")
    ax.bar(x + 0.18, selected["fp64_median"], width=0.36, label="FP64")
    ax.set_yscale("log")
    ax.set_ylabel("relative L2 error")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(p, dpi=180)
    plt.close(fig)
    display(Image(filename=str(p)))

    p = fig_dir / "simple_fp64_fp32_ratio.png"
    fig, ax = plt.subplots(figsize=(max(8, len(selected) * 1.3), 4))
    ax.bar(x, selected["ratio"])
    ax.axhline(1, color="black", linestyle="--", linewidth=1)
    ax.set_yscale("log")
    ax.set_ylabel("FP64 / FP32")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(p, dpi=180)
    plt.close(fig)
    display(Image(filename=str(p)))

    tmp_rows = []
    for _, r in selected.iterrows():
        tmp = df[(df["case"] == r["case"]) & (df["variant"] == r["variant"]) & (df["dtype"].isin(["fp32", "fp64"]))]
        tmp_rows.append(tmp)
    sc = pd.concat(tmp_rows, ignore_index=True)
    pos = {v: i for i, v in enumerate(labels)}
    colors = {"fp32": "#4c78a8", "fp64": "#f58518"}

    p = fig_dir / "simple_seed_scatter.png"
    fig, ax = plt.subplots(figsize=(max(8, len(selected) * 1.3), 4.5))
    for dt in ["fp32", "fp64"]:
        t = sc[sc["dtype"] == dt]
        xs = []
        for v in t["case"]:
            xs.append(pos[v] + (-0.12 if dt == "fp32" else 0.12))
        ax.scatter(xs, t["best_l2_error"], label=dt.upper(), color=colors[dt], s=45)
    ax.set_yscale("log")
    ax.set_ylabel("best relative L2 error")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(p, dpi=180)
    plt.close(fig)
    display(Image(filename=str(p)))

## Кривые обучения

In [ ]:
for _, r in selected.iterrows():
    tmp = df[(df["case"] == r["case"]) & (df["variant"] == r["variant"]) & (df["metrics_path"] != "")]
    if len(tmp) == 0:
        continue
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ok = 0
    for _, run in tmp.sort_values(["dtype", "seed"]).iterrows():
        p = root / run["metrics_path"]
        try:
            h = pd.read_csv(p)
        except Exception:
            continue
        if "step" not in h.columns:
            continue
        label = f"{str(run['dtype']).upper()} seed={int(run['seed']) if pd.notna(run['seed']) else '?'}"
        if "total_loss" in h.columns:
            y = pd.to_numeric(h["total_loss"], errors="coerce")
            m = np.isfinite(y) & (y > 0)
            ax[0].plot(h.loc[m, "step"], y[m], label=label)
        if "l2_error" in h.columns:
            y = pd.to_numeric(h["l2_error"], errors="coerce")
            m = np.isfinite(y) & (y > 0)
            ax[1].plot(h.loc[m, "step"], y[m], label=label)
        ok += 1
    if ok == 0:
        plt.close(fig)
        continue
    ax[0].set_yscale("log")
    ax[1].set_yscale("log")
    ax[0].set_title("loss")
    ax[1].set_title("relative L2")
    ax[0].set_xlabel("step")
    ax[1].set_xlabel("step")
    ax[0].grid(True, alpha=0.3)
    ax[1].grid(True, alpha=0.3)
    ax[0].legend(fontsize=8)
    ax[1].legend(fontsize=8)
    fig.suptitle(r["case"])
    fig.tight_layout()
    name = re.sub(r"[^A-Za-z0-9_]+", "_", r["case"] + "_" + str(r["variant"])).strip("_")
    p = fig_dir / f"simple_curves_{name}.png"
    fig.savefig(p, dpi=180)
    plt.close(fig)
    display(Image(filename=str(p)))

## FP16 и краткое резюме

In [ ]:
fp16 = df[df["dtype"] == "fp16"]
if len(fp16):
    display(Markdown("### FP16"))
    display(fp16.groupby(["case", "variant"]).agg(
        runs=("run_dir", "count"),
        bad=("is_bad", "sum"),
        median_best_l2=("best_l2_error", "median"),
    ).reset_index())
else:
    print("fp16 запусков не найдено")

text = []
text.append("# Report results")
text.append("")
text.append("Основной файл: `results_overview.ipynb`.")
text.append("")
text.append(f"Найдено run-папок с summary.json: {len(df)}.")
text.append(f"Валидных запусков: {int(df['is_valid'].sum())}.")
text.append(f"Плохих/упавших запусков: {int(df['is_bad'].sum())}.")
text.append("")
text.append("Ноутбук намеренно простой: он читает реальные run-папки, берет метрики из metrics.csv, группирует по задаче/параметру/variant/dtype и строит несколько графиков.")
text.append("")
text.append("Главный вывод нужно формулировать осторожно: fp64 иногда помогает в тяжелых режимах, но не всегда лучше fp32; fp16 в этих запусках в основном нестабилен.")
res = "\n".join(text)
(out_dir / "README.md").write_text(res)
display(Markdown(res))